# CMU SCS News: recover incomplete bodies and drop external rows

Two fixes on the raw scrape (`CMU SCS AI Articles.csv`):

1. **Recover 9 missing bodies.** Rows marked `Incomplete` are JavaScript-heavy pages; they are re-scraped with Playwright (`networkidle`) and paragraphs of at least 60 characters are kept.
2. **Drop 8 external-media rows.** Rows with `Official CMU Domain == False` are coverage on third-party sites (Inc., NBC News, ...), not CMU-produced text, and are removed — consistent with the external-redirect rule used for every school.

Output: `./Data/CMU Articles.csv` — the final CMU corpus used by the analysis. (Invalid `$dateFormat` dates are repaired from the listing date inside the analysis notebook.)


In [ ]:
import pandas as pd

RAW_PATH = "./Data/CMU SCS AI Articles.csv"
RECOVERED_PATH = "./Data/CMU Recovered Bodies.csv"
FINAL_PATH = "./Data/CMU Articles.csv"

cmu_df = pd.read_csv(RAW_PATH)
print(f"Raw CMU rows: {len(cmu_df)}")
print(cmu_df["Scrape Status"].value_counts(dropna=False))

## 1. Recover the `Incomplete` rows with Playwright

In [ ]:
from playwright.async_api import async_playwright

targets = cmu_df.loc[cmu_df["Scrape Status"].eq("Incomplete"), "Url"].tolist()
print(f"Pages to recover: {len(targets)}")

async def recover(urls):
    results = []
    async with async_playwright() as pw:
        browser = await pw.chromium.launch()
        page = await browser.new_page()
        for url in urls:
            try:
                await page.goto(url, wait_until="networkidle", timeout=60_000)
                paragraphs = await page.eval_on_selector_all(
                    "p", "ns => ns.map(n => n.innerText.trim()).filter(t => t.length >= 60)"
                )
                body = "\n\n".join(paragraphs)
                results.append({"Url": url, "Recovered Title": await page.title(),
                                "Recovered Body": body, "Recovered Chars": len(body)})
                print(f"OK   {len(body):6d} chars  {url}")
            except Exception as exc:
                results.append({"Url": url, "Recovered Body": "", "Error": str(exc)})
                print(f"FAIL {url}  {exc}")
        await browser.close()
    return pd.DataFrame(results)

recovered = await recover(targets)
recovered.to_csv(RECOVERED_PATH, index=False, encoding="utf-8-sig")
print(f"Saved to: {RECOVERED_PATH}")

## 2. Merge recovered bodies, drop external-media rows, save the final corpus

In [ ]:
recovered = pd.read_csv(RECOVERED_PATH)
recovered = recovered[recovered["Recovered Body"].fillna("").str.len() > 200]

cmu_fixed = cmu_df.merge(recovered[["Url", "Recovered Body"]], on="Url", how="left")
fix_mask = cmu_fixed["Recovered Body"].notna()
cmu_fixed.loc[fix_mask, "Article Body"] = cmu_fixed.loc[fix_mask, "Recovered Body"]
cmu_fixed.loc[fix_mask, "Scrape Status"] = "Success (Playwright recovery)"
cmu_fixed = cmu_fixed.drop(columns=["Recovered Body"])
print(f"Bodies recovered: {int(fix_mask.sum())}")

external_mask = ~cmu_fixed["Official CMU Domain"].fillna(False)
print(f"External-media rows dropped: {int(external_mask.sum())}")
cmu_final = cmu_fixed.loc[~external_mask].reset_index(drop=True)
print(f"Final CMU rows: {len(cmu_final)}")

cmu_final.to_csv(FINAL_PATH, index=False, encoding="utf-8-sig")
print(f"Saved final CMU corpus to: {FINAL_PATH}")